In [1]:
import pandas as pd

matches = pd.read_csv("../data/processed/matches_cleaned.csv")
deliveries = pd.read_csv("../data/processed/deliveries_cleaned.csv")

print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

matches.head()
deliveries.head()

Matches: (891, 20)
Deliveries: (260920, 17)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [2]:
team_runs = deliveries.groupby(
    ['match_id', 'batting_team']
)['total_runs'].sum().reset_index()

team_runs.rename(columns={'total_runs': 'team_runs'}, inplace=True)

team_runs.head()

,match_id,batting_team,team_runs
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Kings XI Punjab,207
4,335984,Delhi Daredevils,132


In [3]:
wickets = deliveries[deliveries['is_wicket'] == 1]

team_wickets = wickets.groupby(
    ['match_id', 'batting_team']
)['is_wicket'].sum().reset_index()

team_wickets.rename(columns={'is_wicket': 'team_wickets'}, inplace=True)

team_wickets.head()

,match_id,batting_team,team_wickets
0,335982,Kolkata Knight Riders,3
1,335982,Royal Challengers Bangalore,10
2,335983,Chennai Super Kings,5
3,335983,Kings XI Punjab,4
4,335984,Delhi Daredevils,1


In [4]:
team_stats = pd.merge(
    team_runs,
    team_wickets,
    on=['match_id', 'batting_team'],
    how='left'
)

team_stats.fillna(0, inplace=True)

team_stats.head()

,match_id,batting_team,team_runs,team_wickets
0,335982,Kolkata Knight Riders,222,3.0
1,335982,Royal Challengers Bangalore,82,10.0
2,335983,Chennai Super Kings,240,5.0
3,335983,Kings XI Punjab,207,4.0
4,335984,Delhi Daredevils,132,1.0


In [ ]:
team1 = team_stats.copy()
team2 = team_stats.copy()

team1.columns = ['match_id', 'team1', 'team1_runs', 'team1_wickets']
team2.columns = ['match_id', 'team2', 'team2_runs', 'team2_wickets']

match_features = pd.merge(team1, team2, on='match_id')

# remove same team pairs
match_features = match_features[match_features['team1'] != match_features['team2']]

match_features.head()

,match_id,team1,team1_runs,team1_wickets,team2,team2_runs,team2_wickets
1,335982,Kolkata Knight Riders,222,3.0,Royal Challengers Bangalore,82,10.0
2,335982,Royal Challengers Bangalore,82,10.0,Kolkata Knight Riders,222,3.0
5,335983,Chennai Super Kings,240,5.0,Kings XI Punjab,207,4.0
6,335983,Kings XI Punjab,207,4.0,Chennai Super Kings,240,5.0
9,335984,Delhi Daredevils,132,1.0,Rajasthan Royals,129,8.0


In [8]:
# ==============================
# STEP 6: MERGE MATCHES + DELIVERY FEATURES
# ==============================

data = pd.merge(matches, match_features, on='match_id')

# 🔥 Fix duplicate column names after merge
data = data.rename(columns={
    'team1_x': 'team1',
    'team2_x': 'team2'
})

# 🔥 Drop unnecessary duplicate columns
data = data.drop(columns=['team1_y', 'team2_y'])

# Check result
print("Columns after merge:")
print(data.columns)

print("\nShape:", data.shape)
data.head()

Columns after merge:
Index(['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2', 'team1_runs', 'team1_wickets',
       'team2_runs', 'team2_wickets'],
      dtype='object')

Shape: (1782, 24)


,match_id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,...,target_runs,target_overs,super_over,method,umpire1,umpire2,team1_runs,team1_wickets,team2_runs,team2_wickets
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,222,3.0,82,10.0
1,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,...,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen,82,10.0,222,3.0
2,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,...,241.0,20.0,N,NaN,MR Benson,SL Shastri,240,5.0,207,4.0
3,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Punjab Kings,Chennai Super Kings,Chennai Super Kings,...,241.0,20.0,N,NaN,MR Benson,SL Shastri,207,4.0,240,5.0
4,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,...,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar,132,1.0,129,8.0


In [17]:
# ==============================
# STEP 7: CREATE LEAK-FREE FEATURES
# ==============================

# Target
data['team1_win'] = (data['winner'] == data['team1']).astype(int)

# ✅ Toss feature
data['toss_winner_is_team1'] = (data['toss_winner'] == data['team1']).astype(int)

# ❌ REMOVE THESE (IMPORTANT)
# run_diff ❌
# wicket_diff ❌

# ✅ Keep only pre-match features
features = data[[
    'team1',
    'team2',
    'venue',
    'toss_winner_is_team1',
    'toss_decision'
]]

target = data['team1_win']

print("Features shape:", features.shape)
features.head()

Features shape: (1782, 5)


,team1,team2,venue,toss_winner_is_team1,toss_decision
0,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1,field
1,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1,field
2,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",0,bat
3,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",0,bat
4,Delhi Capitals,Rajasthan Royals,Feroz Shah Kotla,0,bat


In [18]:
# Convert categorical → numeric
features = pd.get_dummies(features, columns=[
    'team1', 'team2', 'venue', 'toss_decision'
], drop_first=True)

print("Final features shape:", features.shape)
features.head()

Final features shape: (1782, 73)


,toss_winner_is_team1,team1_Delhi Capitals,team1_Gujarat Titans,team1_Kolkata Knight Riders,team1_Lucknow Super Giants,team1_Mumbai Indians,team1_Punjab Kings,team1_Rajasthan Royals,team1_Royal Challengers Bangalore,team1_Sunrisers Hyderabad,...,venue_Shaheed Veer Narayan Singh International Stadium,venue_Sharjah Cricket Stadium,venue_Sheikh Zayed Stadium,venue_St George's Park,venue_Subrata Roy Sahara Stadium,venue_SuperSport Park,venue_Wankhede Stadium,"venue_Wankhede Stadium, Mumbai","venue_Zayed Cricket Stadium, Abu Dhabi",toss_decision_field
0,1,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
1,1,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
2,0,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [14]:
print(data.columns)

Index(['match_id', 'season', 'city', 'date', 'match_type', 'player_of_match',
       'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner',
       'result', 'result_margin', 'target_runs', 'target_overs', 'super_over',
       'method', 'umpire1', 'umpire2', 'team1_runs', 'team1_wickets',
       'team2_runs', 'team2_wickets', 'team1_win', 'run_diff', 'wicket_diff',
       'toss_winner_is_team1'],
      dtype='object')


In [15]:
# Convert categorical → numeric
features = pd.get_dummies(features, columns=['venue', 'toss_decision'], drop_first=True)

print("Final features shape:", features.shape)
features.head()

Final features shape: (1782, 57)


,run_diff,wicket_diff,toss_winner_is_team1,"venue_Arun Jaitley Stadium, Delhi",venue_Barabati Stadium,"venue_Barsapara Cricket Stadium, Guwahati","venue_Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow",venue_Brabourne Stadium,"venue_Brabourne Stadium, Mumbai",venue_Buffalo Park,...,venue_Shaheed Veer Narayan Singh International Stadium,venue_Sharjah Cricket Stadium,venue_Sheikh Zayed Stadium,venue_St George's Park,venue_Subrata Roy Sahara Stadium,venue_SuperSport Park,venue_Wankhede Stadium,"venue_Wankhede Stadium, Mumbai","venue_Zayed Cricket Stadium, Abu Dhabi",toss_decision_field
0,140,-7.0,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,-140,7.0,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
2,33,1.0,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,-33,-1.0,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,3,-7.0,0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [19]:

features.to_csv("../data/feature_store/delivery_features.csv", index=False)
target.to_csv("../data/feature_store/delivery_target.csv", index=False)

print(" Delivery features saved successfully")

 Delivery features saved successfully


In [20]:
features.to_csv("../data/feature_store/final_features.csv", index=False)
target.to_csv("../data/feature_store/final_target.csv", index=False)

print(" Leak-free features saved")

 Leak-free features saved
